# RAPiDock Colab Runner (Google Drive mode)

This notebook mounts Google Drive and reads/writes files directly from your Drive copy of the project.

Expected project location in Drive:
- `/content/drive/MyDrive/ghsr_af3_molecule_screening`

You can change the path in Cell 2 if needed.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#@title 1) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
#@title 2) Configure project paths on Drive
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/ghsr_af3_molecule_screening')  # change if needed
RAPIDOCK_REPO = Path('/content/RAPiDock')

RECEPTOR_PDB = PROJECT_DIR / 'data' / 'ghsr_receptor.pdb'
BATCH_CSV = PROJECT_DIR / 'inputs_rapidock' / 'protein_peptide.csv'
OUTPUT_SINGLE = PROJECT_DIR / 'outputs_rapidock_test'
OUTPUT_BATCH = PROJECT_DIR / 'outputs_rapidock'

print('PROJECT_DIR:', PROJECT_DIR)
print('RECEPTOR_PDB exists:', RECEPTOR_PDB.exists())
print('BATCH_CSV exists:', BATCH_CSV.exists())

PROJECT_DIR: /content/drive/MyDrive/ghsr_af3_molecule_screening
RECEPTOR_PDB exists: False
BATCH_CSV exists: False


In [4]:
#@title 3) Clone RAPiDock
!git clone https://github.com/huifengzhao/RAPiDock.git
%cd /content/RAPiDock

Cloning into 'RAPiDock'...
remote: Enumerating objects: 125, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 125 (delta 48), reused 96 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (125/125), 13.52 MiB | 22.21 MiB/s, done.
Resolving deltas: 100% (48/48), done.
/content/RAPiDock


In [8]:
#@title 4) Install RAPiDock dependencies (Colab/Linux)
!pip install -U pip
!pip install pyyaml pandas biopython MDAnalysis rdkit-pypi fair-esm e3nn pyrosetta-installer
!pip install torch-geometric torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.4.0+cpu.html

  Using cached biopython-1.87-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (13 kB)
  Using cached mdanalysis-2.10.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (108 kB)
ERROR: Ignored the following versions that require a different python version: 1.0.1 Requires-Python !=3.0.*,!=3.1.*,!=3.2.*,!=3.3,!=3.4.*,>=2.7,<3.9; 1.1.0 Requires-Python !=3.0.*,!=3.1.*,!=3.2.*,!=3.3.*,!=3.4.*,>=2.7,<3.9; 1.1.1 Requires-Python !=3.0.*,!=3.1.*,!=3.2.*,!=3.3.*,!=3.4.*,>=2.7,<3.9
ERROR: Could not find a version that satisfies the requirement rdkit-pypi (from versions: none)
ERROR: No matching distribution found for rdkit-pypi
Looking in links: https://data.pyg.org/whl/torch-2.4.0+cpu.html


In [6]:
#@title 5) Download checkpoint
!mkdir -p /content/RAPiDock/train_models/CGTensorProductEquivariantModel
!wget -O /content/RAPiDock/train_models/CGTensorProductEquivariantModel/rapidock_local.pt "https://zenodo.org/records/14193621/files/rapidock_local.pt?download=1"
!ls -lh /content/RAPiDock/train_models/CGTensorProductEquivariantModel/rapidock_local.pt

--2026-04-30 02:53:26--  https://zenodo.org/records/14193621/files/rapidock_local.pt?download=1
Resolving zenodo.org (zenodo.org)... 137.138.153.219, 137.138.52.235, 188.184.98.114, ...
Connecting to zenodo.org (zenodo.org)|137.138.153.219|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 56648773 (54M) [application/octet-stream]
Saving to: ‘/content/RAPiDock/train_models/CGTensorProductEquivariantModel/rapidock_local.pt’

/content/RAPiDock/t 100%[===================>]  54.02M  19.7MB/s    in 2.7s    

2026-04-30 02:53:29 (19.7 MB/s) - ‘/content/RAPiDock/train_models/CGTensorProductEquivariantModel/rapidock_local.pt’ saved [56648773/56648773]

-rw-r--r-- 1 root root 55M Apr 30 02:53 /content/RAPiDock/train_models/CGTensorProductEquivariantModel/rapidock_local.pt


In [7]:
#@title 6) Single-case smoke test to Drive output
!python /content/RAPiDock/inference.py \
  --complex_name hexarelin_test \
  --protein_description "$RECEPTOR_PDB" \
  --peptide_description "H[DTR]AW[DPN]K" \
  --output_dir "$OUTPUT_SINGLE" \
  --model_dir /content/RAPiDock/train_models/CGTensorProductEquivariantModel \
  --ckpt rapidock_local.pt \
  --scoring_function ref2015 \
  --N 1 --batch_size 1 \
  --no_final_step_noise \
  --inference_steps 8 --actual_steps 8 \
  --conformation_partial 1:1:1 \
  --cpu 4

Traceback (most recent call last):
  File "/content/RAPiDock/inference.py", line 13, in <module>
    import MDAnalysis
ModuleNotFoundError: No module named 'MDAnalysis'


In [ ]:
#@title 7) Full batch run to Drive output (optional)
!python /content/RAPiDock/inference.py \
  --protein_peptide_csv "$BATCH_CSV" \
  --output_dir "$OUTPUT_BATCH" \
  --model_dir /content/RAPiDock/train_models/CGTensorProductEquivariantModel \
  --ckpt rapidock_local.pt \
  --scoring_function ref2015 \
  --N 10 --batch_size 4 \
  --no_final_step_noise \
  --inference_steps 16 --actual_steps 16 \
  --conformation_partial 1:1:1 \
  --cpu 8

In [ ]:
#@title 8) Verify Drive outputs
!ls -R "$OUTPUT_SINGLE"
print('---')
!ls -R "$OUTPUT_BATCH"

## Next step (local parsing)

After Colab finishes, go back to your local repo and run:

```bash
python scripts/parse_rapidock_results.py
python scripts/parse_results.py
python scripts/visualize.py
```

Because outputs are written directly to Drive, no manual download/upload step is needed.